## Import libraries and healthiar

In [11]:
## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

## Import pry2 functions
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import FloatVector, IntVector, globalenv, ListVector, DataFrame, StrVector, BoolVector
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface as ri

## Initialise R session in background
ri.initr()

## Import other modules
import pandas as pd
import geopandas as gpd
import matplotlib as plt
import xarray as xr
import rioxarray
import numpy as np
from scipy import interpolate
from types import SimpleNamespace

## Import healthiar as Python wrapper
healthiar = importr("healthiar")

## Define functions

In [51]:
## Function to convert pandas dataframe to R data.frame
def py_to_r(obj):
    if isinstance(obj, (tuple, list, pd.Series)):
        if all(isinstance(item, int) for item in obj):
            return IntVector(obj)
        elif all(isinstance(item, float) for item in obj):
            return FloatVector(obj)
        elif all(isinstance(item, str) for item in obj):
            return StrVector(obj)
        else:
            raise ValueError("Conversion not possible for tuple/list with these element types.")
    elif isinstance(obj, pd.DataFrame):
        with (ro.default_converter + pandas2ri.converter).context():
            return ro.conversion.get_conversion().py2rpy(obj)
    else:
        raise ValueError("Conversion not possible for this object type.")

In [3]:
## Function to convert rpy2 lists and data.frames to python lists and pandas dataframes
def r_to_py(obj):
    """
    Recursive conversion
    """
    if isinstance(obj, DataFrame):
        with localconverter(ro.default_converter + pandas2ri.converter):
            return ro.conversion.rpy2py(obj)
    
    elif isinstance(obj, ListVector):
        return {name: r_to_py(obj.rx2(name)) for name in obj.names}
    
    elif isinstance(obj, (IntVector, FloatVector, StrVector, BoolVector)):
        if len(obj) == 1:
            return obj[0]  # single elements
        else:
            return list(obj)  # multiple elements
    else:
        return obj  # fallback for other types


## Perform tests
### List/tuple input

In [15]:
# Get Python data
exp_central = [20, 20]
prop_pop_exp = [0.5, 0.5]
bhd_central = [10]

# Convert to R data
r_exp_central = py_to_r(exp_central)
r_prop_pop_exp = py_to_r(prop_pop_exp)
r_bhd_central = py_to_r(bhd_central)

# Call healthiar function
result = healthiar.attribute_health(
    exp_central = r_exp_central,
    prop_pop_exp = r_prop_pop_exp,
    cutoff_central = 5,
    rr_central = 1.08,
    rr_increment = 10,
    erf_shape = "linear_log",
    bhd_central = r_bhd_central
)

## Verify result
expected = np.float64(0.927071)
py_result = r_to_py(result)['health_main']['impact'].iloc[0]
py_result = round(py_result, 6)
print(py_result)
print(py_result == expected)

0.927071
True


In [19]:
# Get Python data
exp_central = (20, 20)
prop_pop_exp = (0.5, 0.5)

# Convert to R data
r_exp_central = py_to_r(exp_central)
r_prop_pop_exp = py_to_r(prop_pop_exp)

# Call healthiar function
result = healthiar.attribute_health(
    exp_central = r_exp_central,
    prop_pop_exp = r_prop_pop_exp,
    cutoff_central = 5,
    rr_central = 1.08,
    rr_increment = 10,
    erf_shape = "log_log",
    bhd_central = 10
)

## Verify result
expected = 0.936215963
py_result = r_to_py(result)["health_main"]["impact"].iloc[0]
py_result = round(py_result, 9)
print(py_result)
print(py_result == expected)

0.936215963
True


In [ ]:
# Get Python data
geo_id_micro = ["Zürich", "Basel", "Geneva", "Ticino", "Jura"]
geo_id_macro = ["German","German","French","Italian","French"]
exp_central = [11, 11, 10, 8, 7]
bhd_central = [4000, 2500, 3000, 1500, 500]

# Convert to R data
r_geo_id_micro = py_to_r(geo_id_micro)
r_geo_id_macro = py_to_r(geo_id_macro)
r_exp_central = py_to_r(exp_central)
r_bhd_central = py_to_r(bhd_central)

# Call healthiar function
result = healthiar.attribute_health(
    geo_id_micro = r_geo_id_micro,
    geo_id_macro = r_geo_id_macro,
    erf_shape = "log_linear",
    rr_central = 1.369,
    rr_increment = 10,
    cutoff_central = 5,
    exp_central = r_exp_central,
    bhd_central = r_bhd_central
)

## Verify result
expected = [round(x, 11) for x in [1116.41855325132, 466.433010062623, 134.881901125568]]
py_result = r_to_py(result)["health_main"]["impact"]
py_result = [round(x, 11) for x in py_result]
print(py_result)
print(py_result == expected)

[1116.41855325132, 466.43301006262, 134.88190112557]
True


### pandas DataFrame input

In [40]:
# Get Python data
data = pd.DataFrame({
    "mean_concentration": [8.85],
    "cut_off_value": [5],
    "incidents_per_100_000_per_year": [357.27],
    "population_at_risk": [8606096],
    "relative_risk": [1.369],
    "pollutant": ["PM2.5"],
    "evaluation_name": ["GeLuft_COPD"],
    "estimated_number_of_attributable_cases_central": [3502]
})

# Convert to R data
r_exp_central = py_to_r(data["mean_concentration"])
r_cutoff_central = py_to_r(data["cut_off_value"])
r_bhd_central = py_to_r(data["incidents_per_100_000_per_year"] / 10**5 * data["population_at_risk"])
r_rr_central = py_to_r(data["relative_risk"])

# Call healthiar function
result = healthiar.attribute_health(
    approach_risk = "relative_risk",
    exp_central = r_exp_central,
    cutoff_central = r_cutoff_central,
    bhd_central = r_bhd_central,
    rr_central = r_rr_central,
    rr_increment = 10,
    erf_shape = "log_linear"
)

## Verify result
expected = data["estimated_number_of_attributable_cases_central"]
py_result = r_to_py(result)["health_main"]["impact_rounded"].iloc[0]
print(py_result)
print(py_result == expected)

3502.0
0    True
Name: estimated_number_of_attributable_cases_central, dtype: bool


### Function input

In [52]:
## Get Python data
data = pd.read_csv("data/LMU_O3_COPD_mort_2015_2016.csv", skiprows = [1])

## Convert to R data
@ri.rternalize
def r_erf_eq_central(x):
    cs = interpolate.CubicSpline(data["x"][0:20], data["y"][0:20])
    return float(cs(x)[0])

@ri.rternalize
def r_erf_eq_lower(x):
    cs = interpolate.CubicSpline(data["x"][0:20], data["y_l"][0:20])
    return float(cs(x)[0])

@ri.rternalize
def r_erf_eq_upper(x):
    cs = interpolate.CubicSpline(data["x"][0:20], data["y_u"][0:20])
    return float(cs(x)[0])

r_prop_pop_exp = py_to_r(data["Population.affected"])
r_exp_central = py_to_r(data["Mean.O3"])
r_bhd_central =  py_to_r(data["bhd"])
r_geo_id_micro = py_to_r(data["X"])

## Call healthiar function
result = healthiar.attribute_health(
    erf_eq_central = r_erf_eq_central,
    erf_eq_lower = r_erf_eq_lower,
    erf_eq_upper = r_erf_eq_upper,
    prop_pop_exp = r_prop_pop_exp,
    exp_central = r_exp_central, # exposure distribution for ozone
    cutoff_central = 0,
    bhd_central =  r_bhd_central, #COPD mortality in Germany 2015 and 2016
    geo_id_micro = r_geo_id_micro
)

## Verify result
expected = [350, 267, 424, 313, 238, 379]
py_result = r_to_py(result)["health_main"]["impact_rounded"]

TypeError: 'NULLType' object is not iterable

In [46]:
## Get Python data
data = pd.read_csv("data/roadnoise_ha_Lden_StavangerandVicinity.csv")
info = pd.DataFrame({
    "pollutant": ["road_noise"],
    "outcome": ["highly_annoyance"]
})

## Convert to R data
r_exp_central = py_to_r(data['average_cat'])
r_pop_exp = py_to_r(data['ANTALL_PER'])
r_info = py_to_r(info)

## Call healthiar function
result = healthiar.attribute_health(
    approach_risk = "absolute_risk",
    exp_central = r_exp_central,
    population  = int(data['totpop'][0]),
    pop_exp = r_pop_exp,
    erf_eq_central = "78.9270-3.1162*c+0.0342*c^2",
    info = r_info
)

## Verify result
expected = 14136
py_result = r_to_py(result)["health_main"]["impact_rounded"].iloc[0]
print(py_result)
print(py_result == expected)

New names:
* `0` -> `0...1`
* `0` -> `0...2`
* `0` -> `0...3`
* `0` -> `0...4`
* `0` -> `0...5`
* `0` -> `0...6`
* `0` -> `0...7`
* `0` -> `0...8`
* `0` -> `0...9`
* `0` -> `0...10`
* `0` -> `0...11`
* `0` -> `0...12`
* `0` -> `0...13`
* `0` -> `0...14`
* `0` -> `0...15`
* `0` -> `0...16`
* `0` -> `0...17`
* `0` -> `0...18`
* `0` -> `0...19`
* `0` -> `0...20`
* `0` -> `0...21`
* `0` -> `0...22`
* `0` -> `0...23`
* `0` -> `0...24`
* `0` -> `0...25`
* `0` -> `0...26`
* `0` -> `0...27`
* `0` -> `0...28`
* `0` -> `0...29`
* `0` -> `0...30`
* `0` -> `0...31`
* `0` -> `0...32`
* `0` -> `0...33`
* `0` -> `0...34`
* `0` -> `0...35`
* `0` -> `0...36`
* `0` -> `0...37`
* `0` -> `0...38`
* `0` -> `0...39`
* `0` -> `0...40`
* `0` -> `0...41`
14136.0
True


### Spatial format input

In [49]:
## Get python data
poll_grid = xr.open_dataset("data/pm25.tif", engine = "rasterio", masked = True)
#poll_grid["band_data"].plot()
#print(type(poll_grid))

geo_units = gpd.read_file("data/municipalities_brussels.gpkg")
#geo_units.head()
#geo_units.plot(facecolor = "none")
#print(type(geo_units))

expected = pd.read_csv("data/exp_grid_results.csv")

## Convert to R data
with (ro.default_converter + pandas2ri.converter).context():
  r_geo_units = ro.conversion.get_conversion().py2rpy(geo_units) ## conversion of geometry columns fails

## how to convert xarray Dataset?

## workaround (cheating)
terra = importr("terra")
r_poll_grid = terra.rast("data/pm25.tif")

r_pop_grid = terra.rast("data/population.tif")

sf = importr("sf")
r_geo_units = sf.st_read("data/municipalities_brussels.gpkg", quiet = True)

## Call healthiar function
result = healthiar.prepare_exposure(
  poll_grid = r_poll_grid,
  geo_units = r_geo_units,
  pop_grid = r_pop_grid,
  geo_id_micro = sf.st_drop_geometry(r_geo_units.rx2["name"])
)

## Verify result
expected = list(expected["exposure"])
py_result = r_to_py(result)["exposure_main"]["exposure_mean"]
py_result = [round(x, 13) for x in py_result]
print(py_result)
print(py_result == expected)

c:\Users\ArPa3547\AppData\Local\Programs\Python\Python314\Lib\site-packages\rpy2\robjects\pandas2ri.py:65: UserWarning: Error while trying to convert the column "geometry". Fall back to string conversion. The error is: 'GeometryDtype' object has no attribute 'isnative'
  warnings.warn('Error while trying to convert '


[11.5745782446363, 11.5298189171231, 10.9389592295414, 11.0206189271677, 11.19626729416, 11.0932521442899, 11.3148154965858, 11.135658032178, 11.5235422331439, 11.3523405480154, 11.3252103204793, 11.481438779883, 11.4421057673046, 11.5904140567847, 11.6519109762713, 11.5372396359775, 11.5371125831034, 11.5276393159454, 11.400330254535]
True


### Output reuse

In [50]:
## Get Python data

## Convert to R data
  
## Call healthiar function
output_attribute_scen_1 = healthiar.attribute_health(
  exp_central = 8.85,
  cutoff_central = 5,
  bhd_central = 25000,
  approach_risk = "relative_risk",
  erf_shape = "log_linear",
  rr_central = 1.118, rr_lower = 1.060, rr_upper = 1.179,
  rr_increment = 10,
  info = "PM2.5_mortality_2010"
)

output_attribute_scen_2 = healthiar.attribute_health(
  exp_central = 6,
  cutoff_central = 5,
  bhd_central = 25000,
  approach_risk = "relative_risk",
  erf_shape = "log_linear",
  rr_central = 1.118, rr_lower = 1.060, rr_upper = 1.179,
  rr_increment = 10,
  info = "PM2.5_mortality_2020"
)

result = healthiar.compare(
  output_attribute_scen_1 = output_attribute_scen_1,
  output_attribute_scen_2 = output_attribute_scen_2,
  approach_comparison = "delta"
)

## Verify result
expected = [774, 409, 1127]
py_result = r_to_py(result)["health_main"]["impact_rounded"]
print(py_result)
print(py_result == expected)

1     774.0
2     409.0
3    1127.0
Name: impact_rounded, dtype: float64
1    True
2    True
3    True
Name: impact_rounded, dtype: bool


### Access healthiar documentation

In [ ]:
?healthiar.attribute_health

Signature:       healthiar.attribute_health(*args, **kwargs)
Type:            DocumentedSTFunction
String form:    
function(
           # RR & AR
           approach_risk = "relative_risk",
           exp_central, exp_lower = NULL, e <...> rgs)
           
           return(output)
           
           
           }
           <bytecode: 0x00000230ee920140>
           <environment: namespace:healthiar>
           
File:            c:\users\arpa3547\appdata\local\programs\python\python314\lib\site-packages\rpy2\robjects\functions.py
Docstring:      
Wrapper around an R function.

The docstring below is built from the R documentation.

description
-----------


 This function calculates the attributable health impacts (mortality or morbidity) due to
 exposure to an environmental stressor (air pollution or noise), using either relative risk ( RR ) or absolute risk ( AR ).
 
 Arguments for both  RR & AR  pathways
 
     approach_risk 
     exp_central ,  exp_lower ,  exp_upper 
     cut